# Name That LLM — V1 baseline (submission notebook)

本 Notebook 使用 Kaggle 官方提供的 `train.csv` 训练字符 3–5 gram TF-IDF + LinearSVC，并预测 `test.csv`，生成 `/kaggle/working/submission.parquet`。

本地 GroupKFold 的 OOF Balanced Accuracy = **0.307813**。此数值不是线上榜单成绩。


In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

def locate_competition_file(filename):
    candidates = [
        Path('/kaggle/input/competitions/name-that-llm') / filename,
        Path('/kaggle/input/name-that-llm') / filename,
    ]
    for path in candidates:
        if path.is_file():
            return path
    raise FileNotFoundError(
        'Competition file not found: ' + filename +
        '. Add the Name That LLM competition dataset in Kaggle Notebook.'
    )

train = pd.read_csv(locate_competition_file('train.csv'))
test = pd.read_csv(locate_competition_file('test.csv'))
assert set(['id', 'prompt_id', 'text', 'model']).issubset(train.columns)
assert set(['id', 'text']).issubset(test.columns)
assert train.model.nunique() == 18
assert train[['id','prompt_id','text','model']].notna().all().all()
assert test[['id','text']].notna().all().all()
assert test['id'].is_unique
print('train:',train.shape,'test:',test.shape)

In [ ]:
model = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char', ngram_range=(3,5), min_df=2,
        max_features=100000, sublinear_tf=True
    )),
    ('classifier',LinearSVC(C=1.0,max_iter=4000)),
])
model.fit(train['text'], train['model'])
pred = model.predict(test['text'])
submission = pd.DataFrame({'id':test['id'],'model':pred})
assert submission['id'].equals(test['id'])
assert submission['model'].isin(train['model'].unique()).all()
assert submission[['id','model']].notna().all().all()
assert submission['id'].is_unique
outfile = Path('/kaggle/working/submission.parquet') if Path('/kaggle/working').is_dir() else Path('submission.parquet')
submission.to_parquet(outfile,index=False)
print('Saved:',outfile, 'rows:',len(submission))
print(submission.head())